# LunaX Module 04 — Feature / Descriptor Matching

Turns two Module-3 terrain feature sets into **candidate correspondences**.
No RANSAC here — geometric verification is Module 05.

Pipeline: attach descriptors to semantic features → per-type KNN (BF / FLANN) → Lowe ratio →
mutual consistency → robust scale-ratio filter → one-to-one → `MatchResult` (ids, coords, scores, diagnostics).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root (module_3.py / module_4.py)
os.chdir("..")

import numpy as np
import matplotlib.pyplot as plt
from module_3 import TerrainFeatureExtractor, ImagePreprocessor
from module_4 import (create_matching_config, TerrainFeatureSet, extract_feature_set,
                      match_descriptors, ratio_test, mutual_consistency,
                      match_terrain_features, visualize_matches,
                      make_synthetic_pair, evaluate_against_ground_truth, compute_repeatability,
                      run_self_tests)

## 1. Synthetic self-tests (descriptor primitives + synthetic terrain pair)

In [ ]:
results = run_self_tests()
assert all(results.values())

## 2. Real lunar image vs. a known synthetic transform of itself
Rotation 25°, scale 0.8, shift (30,-20), brightness gain 1.25, gamma 0.75, noise σ=5 — so we have exact ground truth.

In [ ]:
pre = ImagePreprocessor()
raw_a = pre.normalize(pre.load("lunar_samples/sample_41.png"))
raw_b, M = make_synthetic_pair(raw_a, rotation_deg=25, scale=0.8, translation=(30, -20),
                               brightness_gain=1.25, gamma=0.75, noise_sigma=5)

extractor = TerrainFeatureExtractor(onnx_crater_model_path="models/crater_unet.onnx", max_total_features=3000)
set_a, enh_a = extract_feature_set(raw_a, extractor)   # Module 3 + SIFT descriptors at semantic features
set_b, enh_b = extract_feature_set(raw_b, extractor)
print("A:", set_a.count_by_type()); print("B:", set_b.count_by_type())

In [ ]:
config = create_matching_config()   # BF, ratio 0.75 (looser per-type for semantic), mutual, scale filter
result = match_terrain_features(set_a, set_b, config)
d = result.diagnostics
print(d["total_matches"], d["matches_by_type"], "median scale ratio:", round(d["median_scale_ratio"], 3))
for g, gd in d["groups"].items():
    print(f"{g:8s} A={gd['candidates_a']:5d} B={gd['candidates_b']:5d} stages={gd['stages']}")

In [ ]:
ev = evaluate_against_ground_truth(result, M, tolerance_px=4.0)
rep = compute_repeatability(set_a, set_b, M, tolerance_px=4.0)
print(f"precision={ev['precision']:.3f} ({ev['correct']}/{ev['n']}), median err={ev['median_error_px']:.2f}px")
for t in ("crater", "ridge", "texture", "sift"):
    e = ev["by_type"].get(t, {"n": 0, "correct": 0}); r = rep[t]
    print(f"{t:8s} matched {e['correct']:4d}/{e['n']:4d}   detector-repeatable {r['repeatable']:4d}/{r['n_a']:4d}")

In [ ]:
fig = visualize_matches(enh_a, enh_b, result, max_draw=300, title="candidates by type"); plt.show()
proj = np.hstack([result.points_a, np.ones((len(result), 1))]) @ M.T
mask = np.linalg.norm(proj - result.points_b, axis=1) <= 4.0
fig = visualize_matches(enh_a, enh_b, result, max_draw=300, correct_mask=mask, title="ground-truth check"); plt.show()

## 3. Configuration sweep — how each filter changes candidate count and precision

In [ ]:
variants = {
    "ratio only":            {"mutual": False, "scale_constraint": False},
    "ratio + mutual":        {"scale_constraint": False},
    "ratio + mutual + scale": {},
    "FLANN":                 {"matcher": "FLANN"},
    "SIFT only":             {"match_types": ["sift"]},
    "no type constraint":    {"type_constraint": False},
}
for name, over in variants.items():
    r = match_terrain_features(set_a, set_b, over)
    e = evaluate_against_ground_truth(r, M, 4.0)
    print(f"{name:24s} n={len(r):4d} precision={e['precision']:.3f} median_err={e['median_error_px'] if e['n'] else float('nan'):.2f}px")

## 4. Low-level primitives (for Module 05 / future learned descriptors)

In [ ]:
# match_descriptors works on any (N, D) array: float32 -> L2, uint8 -> Hamming
da = np.stack([set_a.descriptor_for(i) for i in set_a.indices_of_type("sift")])
db = np.stack([set_b.descriptor_for(i) for i in set_b.indices_of_type("sift")])
knn_ab = match_descriptors(da, db, method="BF", k=2)
knn_ba = match_descriptors(db, da, method="BF", k=2)
good_ab, good_ba = ratio_test(knn_ab, 0.75), ratio_test(knn_ba, 0.75)
mutual = mutual_consistency(good_ab, good_ba)
print(f"knn={len(knn_ab)} ratio={len(good_ab)} mutual={len(mutual)}")

## 5. Save outputs (JSON consumed by Module 05)

In [ ]:
os.makedirs("module4_outputs", exist_ok=True)
result.save("module4_outputs/sample_41_synthetic_matches.json")
print(result.points_a.shape, result.points_b.shape, result.scores[:5])